# EDA On BDB2026 Data

The 2026 BDB had the following task: *Describe player movement with the ball in the air in a way that is accessible to coaches and football fans.*

The 2026 BDB gave player tracking data for a lot of pass plays from the 2023 season.

***Before Pass Data***: Contains tracking data and some other play/player data. One row per 10Hz period.

***Supplemental Data***: Contains info about the game/play.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    BDB_2026_BEFORE_PASS_DIR,
    BDB_2026_SUPPLEMENTARY_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
from pathlib import Path 

<br><br/>
## Load Data
***

#### Before Pass Data

In [ ]:
# ====================================
# Load Before Pass Data
# ====================================

data_dir = Path(str(BDB_2026_BEFORE_PASS_DIR))

# ------------------------------------
# Load and Combine 18 Before Pass Datasets Into 1
# ------------------------------------

before_pass_df = (
    pl.concat(
        [
            pl.scan_csv(data_dir / f"input_2023_w{i:02d}.csv")
            .with_columns(pl.lit(i).alias("week"))
            for i in range(1, 19)
        ],
        how="vertical"
    )
    .collect()
)

# ------------------------------------
# Add Game-Play and Player-Play ID
# ------------------------------------

before_pass_df = (
    before_pass_df
    .with_columns(
        game_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String), 
        player_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String) + "_" + pl.col("nfl_id").cast(pl.String)
    )
)

print("Before Pass Data (2023 REG Season):")
before_pass_df

<br><br/>
#### Supplemental Data

In [ ]:
# ==================================
# Load Supplemental Data
# ==================================

supp_data = (
    pl.read_csv(
        str(BDB_2026_SUPPLEMENTARY_DIR / "supplementary_data.csv"), 
        # Change "NA" to None
        null_values = {
            "play_action": "NA", 
            "dropback_distance": "NA"
        }
    )
)

print("Supplemental Data:")
supp_data.filter(pl.col("game_id") == 2023090700)

<br><br/>
## Inspect Before-Pass Data
***

In [ ]:
# ==========================================
# Number of Games, Player-Plays, Etc.
# ==========================================

# --------------------------------
# Number of Games
# --------------------------------

num_games = before_pass_df.select("game_id").n_unique()
print(f"    - Number of Games: {num_games}")

# --------------------------------
# Number of Plays
# --------------------------------
num_plays = before_pass_df.select("game_play_id").n_unique()
print(f"    - Number of Plays: {num_plays}")

# --------------------------------
# Number of Player-Plays
# --------------------------------
num_player_plays = before_pass_df.select("player_play_id").n_unique()
print(f"    - Number of Player-Plays: {num_player_plays}")

# --------------------------------
# Average Play Length
# --------------------------------
min_play_secs = round(before_pass_df.group_by("game_play_id").agg(play_length = pl.max("frame_id")).select(pl.min("play_length")).item() / 10, 2)
max_play_secs = round(before_pass_df.group_by("game_play_id").agg(play_length = pl.max("frame_id")).select(pl.max("play_length")).item() / 10, 2)
avg_play_secs = round(before_pass_df.group_by("game_play_id").agg(play_length = pl.max("frame_id")).select(pl.mean("play_length")).item() / 10, 2)
print(f"    - Average Length of a Play (In Seconds): {avg_play_secs} [Min = {min_play_secs}, Max = {max_play_secs}]")

In [ ]:
# =========================================
# Number of Player-Plays By Position
# =========================================

(
    before_pass_df
    .group_by("player_play_id")
    .agg(position = pl.first("player_position"))
    .group_by("position")
    .agg(num_player_plays = pl.len())
    .sort("num_player_plays", descending = True)
    .head(10)
)

In [ ]:
# ======================================
# Player Roles
# ======================================

(
    before_pass_df
    .group_by("player_play_id")
    .agg(player_role = pl.first("player_role"))
    .group_by("player_role")
    .agg(num_player_plays = pl.len())
    .sort("num_player_plays", descending = True)
)

<br><br/>
## Inspect Supplemental Data
***

In [ ]:
# =======================================
# Variables That Could Be Useful
# =======================================

# route of targeted receiver
#   - route ran by targeted receiver
#   - maybe we only want to look at routes with cuts?

# team_coverage_man_zone
#   - was the defense in MAN coverage or ZONE coverage
#   - would only want to look at man coverage (we want a 1-on-1 for WR vs CB)

# team_coverage_type
#   - more detail on what type of defensive scheme the defense was running

# pass_length
#   - length of pass 
#   - maybe we don't want to include super short passes?

# yardline_number
#   - what yardline was the ball on
#   - maybe we don't want to include passes on the goal line?

# pass_result
#   - complete, incomplete, intercepted, sacked, scramble

# play_nullified_by_penalty
#   - did a penalty on the play cancel the play outcome
#   - maybe we don't want to include plays where a penatly occurred?

supp_data.head()

In [ ]:
# ======================================
# Route of Targeted Receiver
# ======================================

targeted_receivers_routes = supp_data.select("route_of_targeted_receiver").unique().to_series().to_list()

print("Targeted Receiver Routes:")
print(f"    - N = {len(targeted_receivers_routes)}")
print(f"    - {sorted(targeted_receivers_routes)}")

(
    supp_data
    .group_by("route_of_targeted_receiver")
    .agg(num_plays = pl.len())
    .sort("num_plays", descending = True)
    .head(10)
)

In [ ]:
# ======================================
# Man or Zone Coverage
# ======================================

(
    supp_data
    .group_by("team_coverage_man_zone")
    .agg(num_plays = pl.len())
    .sort("num_plays", descending = True)
)

In [ ]:
# ======================================
# Pass Result
# ======================================

(
    supp_data
    .group_by("pass_result")
    .agg(num_plays = pl.len())
    .sort("num_plays", descending = True)
)

In [ ]:
# ======================================
# Play Nullified By Penalty
# ======================================

(
    supp_data
    .group_by("play_nullified_by_penalty")
    .agg(num_plays = pl.len())
    .sort("num_plays", descending = True)
)